In [ ]:
from platform import python_version
print(python_version())

In [ ]:
!echo $CONDA_DEFAULT_ENV

### Install Cellpose-SAM

https://pypi.org/project/cellpose/

https://github.com/MouseLand/cellpose/blob/main/notebooks/run_Cellpose-SAM.ipynb

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

In [ ]:
# !pip3 install scikit-learn
# !pip3 install seaborn
# !pip3 install plotly

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(1, '../src/')

from Basic import *
from image_lib import *

pd.set_option("display.precision", 3)
from IPython.display import display, HTML
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

#### Check if colab notebook instance has GPU access

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

In [ ]:
print(">>> PyTorch version", torch.__version__)

if torch.cuda.is_available():
    print(">> current_device:", torch.cuda.current_device())
    print(">> Device:", torch.cuda.get_device_name(0))
    print(">> CUDA:", torch.version.cuda)
else:
    print(">>> only CPU")
    

In [ ]:
root0 = "../../colaboracoes/deOcesano"
os.listdir(root0)

In [ ]:
cp = Cellpose(root0=root0, verbose=True)

In [ ]:
cp.root_samples

In [ ]:
cp.root_crop

In [ ]:
plates = os.listdir(cp.root_samples)
plates

In [ ]:
i=0
plate=plates[i]
cp.create_roots_plate(plate, verbose=True)

plate

In [ ]:
exps = [x for x in os.listdir(cp.root_plate) if os.path.isdir(os.path.join(cp.root_plate, x))]
exps

In [ ]:
j=2
experiment = exps[j]

cp.create_roots_experiment(experiment, verbose=True)

experiment

### Data origin

In [ ]:
i=1
plate = plates[i]
cp.create_roots_plate(plate, verbose=False)

exps = [x for x in os.listdir(cp.root_plate) if os.path.isdir(os.path.join(cp.root_plate, x))]

print(plate, '\n', exps)

In [ ]:
cp.root_image

In [ ]:
fnames = [x for x in os.listdir(cp.root_image) if x.endswith('tif')]
fnames[0]

In [ ]:
i=0
filefig = os.path.join(cp.root_image, fnames[i])
img = io.imread(filefig)

fig = plt.figure(figsize=(5,5))
(width, height, channels) = img.shape
print(width, height)
plt.imshow(img);

In [ ]:
img.shape

In [ ]:
image = cp.read_PIL_image(fnames[i], cp.root_image).convert('RGB')
# image

In [ ]:
type_image = 'tif'; num_test_imgs = 15

dir_origins = ['10% SFB', '1% SFB', '1% SFB and IL-1B']
class_names = ['10pct', '1pct', '1pct_il1b']
dir_names = ['10perc', '1perc', 'IL1B']

root_data, dic_root_train, dic_root_test = \
cp.set_data_origin_and_create_roots_to_train_and_test(crop_segment='crop', class_names=class_names, dir_origins=dir_origins, dir_names=dir_names, verbose=True)

In [ ]:
cp.clean_data(verbose=True)

In [ ]:
cp.root_data

### Sampling

In [ ]:
root_train_case = os.path.join(cp.root_train)
root_test_case = os.path.join(cp.root_test)

In [ ]:
max_images=1200
perc_train = .60
perc_test = 1-perc_train

dic={}; icount=-1
for i in range(len(class_names)):
    root_data_case = os.path.join(cp.root_data, dir_origins[i])
    class_name = class_names[i]

    fnames = [x for x in os.listdir(root_data_case) if x.endswith('png')]
    random.shuffle(fnames)
    
    n = len(fnames)

    samples = fnames[:max_images]
    n_samples = len(samples)

    n_train_samples = int(n_samples*perc_train)
    train_samples = samples[:n_train_samples]
    test_samples  = samples[n_train_samples:]

    n_test_samples  = len(test_samples)

    icount += 1
    
    dic[icount] = {}
    dic2 = dic[icount]
    dic2['class_name'] = class_name
    dic2['n'] = n
    dic2['perc_train'] = perc_train
    dic2['n_samples'] = n_samples
    dic2['n_train_samples'] = n_train_samples
    dic2['n_test_samples'] = n_test_samples
    dic2['root_data'] = root_data_case
    dic2['train_samples'] = train_samples
    dic2['test_samples'] = test_samples
    
    print(f"{i}) {class_name:12} n={n:5} -> {root_data_case}")

df = pd.DataFrame(dic).T
df

In [ ]:
def copy_data_train_test(class_names:List, dir_origins:List, dir_names:List, max_images:int=1200, perc_train:float=.60, 
                         type_image:str='png', verbose:bool=False) -> Tuple[bool, pd.DataFrame]:

    if not cp.clean_data(verbose=verbose):
        return False, pd.DataFrame()
    
    perc_test = 1-perc_train
    
    dic={}; icount=-1
    for i in range(len(class_names)):
        class_name = class_names[i]
        dir_origin = dir_origins[i]
        dir_target = dir_names[i]

        root_data_case = os.path.join(cp.root_data, dir_origin)
        root_target_train = create_dir(cp.root_train, dir_target)
        root_target_test  = create_dir(cp.root_test,  dir_target)
    
        fnames = [x for x in os.listdir(root_data_case) if x.endswith(type_image)]
        
        n = len(fnames)
        maxi = max_images

        if n > maxi:
            maxi = n
    
        random.shuffle(fnames)
        samples = fnames[:max_images]
        n_samples = len(samples)
    
        n_train_samples = int(n_samples*perc_train)
        train_samples = samples[:n_train_samples]
        test_samples  = samples[n_train_samples:]
    
        n_test_samples  = len(test_samples)
    
        icount += 1
        
        dic[icount] = {}
        dic2 = dic[icount]
        dic2['class_name'] = class_name
        dic2['n'] = n
        dic2['perc_train'] = perc_train
        dic2['n_samples'] = n_samples
        dic2['n_train_samples'] = n_train_samples
        dic2['n_test_samples'] = n_test_samples
        dic2['root_data'] = root_data_case
        dic2['root_target_train'] = root_target_train
        dic2['root_target_test'] = root_target_test
        dic2['train_samples'] = train_samples
        dic2['test_samples'] = test_samples
        
        print(f"{i}) {class_name:12} n={n:5} -> {root_data_case}")

    df = pd.DataFrame(dic).T

    for i in range(len(df)):
        row = df.iloc[i]
        class_name = row.class_name
        root_data = row.root_data

        #------------ train data -------------------------------
        root_target_train = row.root_target_train
        train_samples = row.train_samples
        train_samples = train_samples if isinstance(train_samples, list) else eval(train_samples)
        
        print(f">>> moving class '{class_name}': {len(train_samples)} train samples from \n{root_data} to {root_target_train}")
        for fname in train_samples:
            filename = os.path.join(root_data, fname)
            try:
                shutil.copy(filename, root_target_train)
            except:
                print(f"Error: moving {filename} to {root_target_train}")
                return False, df
                      

        #------------ test data --------------------------------
        root_target_test  = row.root_target_test
        test_samples = row.test_samples
        test_samples = test_samples if isinstance(test_samples, list) else eval(test_samples)
        
        print(f">>> moving class '{class_name}': {len(test_samples)} test samples from \n{root_data} to {root_target_test}")
        for fname in test_samples:
            filename = os.path.join(root_data, fname)
            try:
                shutil.copy(filename, root_target_test)
            except:
                print(f"Error: moving {filename} to {root_target_test}")
                return False, df

    return True, df


In [ ]:
class_names, dir_origins, dir_names

In [ ]:
ret, df = copy_data_train_test(class_names=class_names, dir_origins=dir_origins, dir_names=dir_names, max_images=1200, perc_train=.60, type_image='png')
print(ret)

In [ ]:
df

In [ ]:
print(df.iloc[0].root_data)
print(df.iloc[0].root_target_train)
print(df.iloc[0].root_target_test)

### ImageDataset

In [ ]:
img.shape

In [ ]:
ncrop = 5
size_x = int(width/5)
size_y = int(height/5)
size_x, size_y, 3

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(size=(size_x, size_y)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(size=(size_x, size_y)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
df.columns

In [ ]:
class ImageDataset(torch.utils.data.Dataset):

    def __init__(self, df:pd.DataFrame, transform, train_or_test:str='train'):
        
        self.dic_images, self.dic_image_dirs = {}, {}
        self.transform = transform
        self.train_or_test = train_or_test

        self.class_names = df.class_name.to_list()
        
        for i in range(len(df)):
            row = df.iloc[i]

            class_name = row.class_name

            if train_or_test == 'train':
                samples = row.train_samples
                root = row.root_target_train
            elif train_or_test == 'test':
                samples = row.test_samples
                root = row.root_target_test
            else:
                print("Error: train or test?")
                raise ValueError('\n\n--------------- stop --------------\n')

            samples = samples if isinstance(samples, list) else eval(samples)

            self.dic_images[class_name] = samples
            self.dic_image_dirs[class_name] = root

    def __len__(self):
        return sum([len(self.dic_images[class_name]) for class_name in self.class_names])
    
    
    def __getitem__(self, index):
        class_name = random.choice(self.class_names)
        index = index % len(self.dic_images[class_name])
        image_name = self.dic_images[class_name][index]
        image_path = self.dic_image_dirs[class_name]
        
        # image = Image.open(filename).convert('RGB')
        image = cp.read_PIL_image(image_name, image_path).convert('RGB')
        return self.transform(image)

# Prepare DataLoader

In [ ]:
train_dataset = ImageDataset(df=df, transform=train_transform, train_or_test='train')

In [ ]:
len(train_dataset.dic_images), list(train_dataset.dic_images.keys())

In [ ]:
test_dataset = ImageDataset(df=df, transform=test_transform, train_or_test='test')

In [ ]:
test_dataset.dic_images.keys()

In [ ]:
# test_dataset.dic_images['1pct']

In [ ]:
batch_size = 60

dl_train = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dl_test  = torch.utils.data.DataLoader(test_dataset,  batch_size=batch_size, shuffle=True)

print('Number of training batches', len(dl_train))
print('Number of test batches', len(dl_test))

In [ ]:
for x in dl_train:
    print(x.shape, x)
    break

In [ ]:
x = next(iter(dl_train))
x

# Creating the Model

In [ ]:
train_dataset.class_names

### Parameters

In [ ]:
lr=1e-4
n_classes = len(class_names)


### Model: ResNet18

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(in_features=model.fc.in_features, out_features= n_classes)

try:
    device = "cuda"
    model = model.to(device)
except:
    print("Error: device must be CUDA - however, it is not available")

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

from torch.optim.lr_scheduler import StepLR
# scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.1)
scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
for x in dl_test:
    print(x[0][0])
    break

In [ ]:
def show_preds():
    model.eval()
    images, labels = next(iter(dl_test))
    outputs = model(images)
    _, preds = torch.max(outputs, 1)
    print(len(images), labels, preds)
    show_images(images, labels, preds)

In [ ]:
show_preds()

# Training the Model

In [ ]:
root_data

In [ ]:
root_best = os.path.join(root_data, 'best_model')
create_dir(root_best)
fname_best_model= os.path.join(root_best, 'best_model_weights.pth')

fname_best_model

In [ ]:
def train(epochs):
    print('Starting training..')
    train_loss_list = []; mean_accuracy_list=[]; lr_list=[]
    count_resnet=-1; dic= {}; max_accuracy = 0; min_loss=9999

    for e in range(epochs):
        print('='*20)
        print(f'Starting epoch {e + 1}/{epochs}')
        print('='*20)

        train_loss = 0.
        val_loss = 0.

        resnet18.train() # set model to training phase
            
        accuracy_list = []

        for nTrain, (images, labels) in enumerate(dl_train):
            optimizer.zero_grad()
            outputs = resnet18(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
            if nTrain % 10 == 0:
                print('Evaluating at step', nTrain)

                accuracy = 0
                resnet18.eval() # set model to eval phase
                for val_step, (images, labels) in enumerate(dl_test):
                    outputs = resnet18(images)
                    loss = loss_fn(outputs, labels)
                    val_loss += loss.item()

                    _, preds = torch.max(outputs, 1)
                    accuracy += np.sum((preds == labels).numpy())

                val_loss /= (val_step + 1)
                accuracy = accuracy/len(test_dataset)
                accuracy_list.append(accuracy)
                lr=optimizer.param_groups[0]["lr"]
                
                print(f'Validation Loss: {val_loss:.4f}, Accuracy: {accuracy:.4f}, LR: {lr:.2e}')

                # show_preds()

                resnet18.train()

                if accuracy >= 0.80:
                    loss = train_loss / (nTrain + 1)
                    
                    if (accuracy >= max_accuracy) or (loss < min_loss):
                        
                        if (accuracy >= max_accuracy):
                            ''' PyTorch models store the learned parameters in an internal state dictionary, 
                                called state_dict. These can be persisted via the torch.save method '''
                            torch.save(resnet18.state_dict(), fname_best_model)

                        max_accuracy = accuracy
                        min_loss = loss
                        resnetx = resnet18.state_dict()
                        count_resnet += 1
                        dic[count_resnet]  = {}
                        dic[count_resnet]['lr'] = np.log10(lr)
                        dic[count_resnet]['accuracy'] = accuracy
                        dic[count_resnet]['loss'] = loss
                        dic[count_resnet]['resnet'] = resnetx
                        print(">>> %d) max accuracy %.2f, min loss %.5f "%(count_resnet, max_accuracy*100, min_loss) )

        mean_accuracy = np.mean(accuracy_list)
        train_loss /= (nTrain + 1)
        
        lr_list.append(lr)

        print(f'Training Loss: {train_loss:.4f}')
        train_loss_list.append(train_loss)
        mean_accuracy_list.append(mean_accuracy)
        
        scheduler.step()
        
    print('Training complete..')
    return train_loss_list, mean_accuracy_list, lr_list, dic

In [ ]:
%%time
train_loss_list, mean_accuracy_list, lr_list, dic = train(epochs=50)

### Data augmentation & transforms

### Load dataset (ImageFolder)

In [ ]:
# Folder structure:
# data/train/class1/*.jpg
# data/train/class2/*.jpg
# --------------------------------------------------
train_ds = datasets.ImageFolder("data/train", transform=transform_train)
test_ds  = datasets.ImageFolder("data/test",  transform=transform_test)

### Load ResNet18 + modify last layer

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

num_classes = len(train_ds.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)


### Training setup

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
%%time

for epoch in range(10):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}")



In [ ]:
model_name = 'resnet18'
segment_crop = f'_crop_{ncrop}'

fname = f"model_{model_name}_classifier_{segment_crop}.pt"

try:
    torch.save(model.state_dict(), fname)
except:
    print("Error: could not save the model")

### Evaluation

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct/total:.2f}%")